# 🌙 Aula 18+19 — Concorrência e Energia na GPU

**Curso:** Técnico em Inteligência Artificial | **UC:** Introdução a Arquitetura de Computadores

**Objetivo:** operar a GPU com **justiça** (1 job por vez + fila com prioridade) e com
**eficiência** (medir e limitar a energia). O notebook reúne os temas de **concorrência** e
**energia** num roteiro único.

**Roteiro:**
1. O problema: 4 jobs em 1 GPU (race condition / OOM).
2. **Parte 1 — Concorrência:** lock por diretório + fila por prioridade (Prática 1).
3. **Parte 2 — Energia:** TDP, throttling, Power Limit e eficiência (Prática 2).
4. Integração, síntese e **Exercícios (5)**.

> 💡 **Sem GPU NVIDIA?** Tudo roda em **modo simulado** — os conceitos valem igual. Em
> servidor Linux + NVIDIA, os sensores e o `flock` são reais.


## 1. Diagnóstico do ambiente

Vamos ver o que existe aqui: GPU NVIDIA? `flock`? No Colab não há nenhum dos dois — por isso
usamos **lock por diretório** e dados **simulados** para a energia.


In [ ]:
# @title 🔍 GPU, nvidia-smi e flock disponiveis?
# ============================================================================
# OBJETIVO: descobrir o que o ambiente oferece para adaptar as demonstracoes.
# ============================================================================
import shutil, subprocess

TEM_NVIDIA = shutil.which("nvidia-smi") is not None
TEM_FLOCK = shutil.which("flock") is not None

print(f"nvidia-smi : {'sim' if TEM_NVIDIA else 'nao (usaremos modo simulado)'}")
print(f"flock      : {'sim' if TEM_FLOCK else 'nao (usaremos lock por diretorio)'}")

if TEM_NVIDIA:
    print(subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,power.draw,power.limit,temperature.gpu",
         "--format=csv,noheader"], capture_output=True, text=True).stdout)


## Parte 1 — Concorrência

### 2. O problema: 4 jobs, 1 GPU

Sem controle, os jobs competem pela VRAM ao mesmo tempo e causam **CUDA OOM**. A GPU é um
recurso **exclusivo**: garantimos **um job por vez** com um **lock**.


In [ ]:
# @title 💥 Simulacao da race condition (sem lock)
# ============================================================================
# OBJETIVO: mostrar que, sem lock, varios jobs alocam ao mesmo tempo e estouram
# a memoria. A GPU tem 4 GB no exemplo; cada job pede 2.5 GB.
# ============================================================================
VRAM_TOTAL = 4.0   # GB
VRAM_USADA = 0.0

def alocar(nome, gb):
    """Simula um job pedindo memoria da GPU."""
    global VRAM_USADA
    if VRAM_USADA + gb > VRAM_TOTAL:
        print(f"[{nome}] CUDA OOM! (pediu {gb} GB, restam {VRAM_TOTAL - VRAM_USADA:.1f} GB)")
        return False
    VRAM_USADA += gb
    print(f"[{nome}] alocou {gb} GB (total {VRAM_USADA:.1f}/{VRAM_TOTAL} GB)")
    return True

print("=== SEM LOCK: 4 jobs ao mesmo tempo ===")
for job in ["A", "B", "C", "D"]:
    alocar(f"Job-{job}", 2.5)
print("\nConclusao: sem exclusao mutua, os jobs se destroem (OOM).")


In [ ]:
# @title 🔒 Lock por diretorio (exclusao mutua) — funciona no Colab/Windows
# ============================================================================
# OBJETIVO: garantir que apenas UM job use a GPU por vez.
# os.mkdir e ATOMICO: se o diretorio ja existe, falha -> outro ja tem o lock.
# ============================================================================
import os, time, shutil

LOCK_DIR = "gpu.lockdir"
shutil.rmtree(LOCK_DIR, ignore_errors=True)   # limpa estado anterior

def tentar_lock():
    """Tenta adquirir o lock. True se conseguiu, False se ja estava ocupado."""
    try:
        os.mkdir(LOCK_DIR)      # atomico: so um processo consegue
        return True
    except FileExistsError:
        return False

def liberar_lock():
    shutil.rmtree(LOCK_DIR, ignore_errors=True)

print("Job-A tentando o lock...", tentar_lock())
print("Job-B tentando o lock...", tentar_lock(), "(esperado: False -> GPU ocupada)")
liberar_lock()
print("Job-A liberou. Job-B tentando de novo...", tentar_lock())
liberar_lock()
print("\nConclusao: com o lock, um job espera o outro -> sem OOM.")


In [ ]:
# @title 📋 Fila por prioridade (ticket = prioridade_timestamp_nome)
# ============================================================================
# OBJETIVO: em vez de corrida livre, os jobs entram numa fila e executam em
# ordem de prioridade. O ticket 'prioridade_timestamp_nome' ordena via sort.
# ============================================================================
import os, shutil, datetime

SPOOL = "gpu_queue_spool"
shutil.rmtree(SPOOL, ignore_errors=True)
os.makedirs(SPOOL)

def enfileirar(prioridade, nome):
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    ticket = f"{prioridade}_{ts}_{nome}"
    open(os.path.join(SPOOL, ticket), "w").close()
    print(f"  enfileirado: {ticket}")

enfileirar(1, "Job-Alta-A")
enfileirar(3, "Job-Baixa-B")
enfileirar(2, "Job-Media-C")
enfileirar(1, "Job-Alta-D")

print("\nOrdem de execucao (sort lexicografico):")
for t in sorted(os.listdir(SPOOL)):
    print("  ->", t)
print("\nPrioridade 1 roda antes de 2, que roda antes de 3.")


### Prática 1 — o experimento completo

Agora simulamos os 4 jobs passando pelo **lock + fila**: cada um espera a vez e executa a
GPU com exclusividade. Observe que **nenhum** resulta em OOM.


In [ ]:
# @title ▶️ Pratica 1: 4 jobs serializados por prioridade
# ============================================================================
# OBJETIVO: percorrer a fila em ordem, adquirindo o lock antes de cada job.
# ============================================================================
import time

fila = sorted(os.listdir(SPOOL))
for ticket in fila:
    prioridade, ts, *resto = ticket.split("_", 2)
    nome = resto[0] if resto else "job"
    while not tentar_lock():
        time.sleep(0.1)           # espera a vez
    print(f"[{time.strftime('%H:%M:%S')}] Executando {nome} (prio={prioridade})")
    time.sleep(0.3)               # 'treino' simulado
    liberar_lock()

print("\nTodos os jobs concluidos, um por vez, sem OOM!")


## Parte 2 — Energia

### 3. TDP, throttling e o custo escondido

**TDP/TGP** é o teto de consumo da placa. Acima de ~83–87 °C a GPU faz **thermal throttling**
(reduz o clock sozinha): perde desempenho e gasta mais. GPUs no TDP máximo em baixa carga são
a causa clássica de contas 40% acima do projetado.


In [ ]:
# @title 🌡️ Modelo termico simplificado (throttling)
# ============================================================================
# OBJETIVO: mostrar que acima de ~85 C a GPU reduz o clock automaticamente.
# ============================================================================
def clock_efetivo(temp_c, clock_base=1410):
    """Devolve o clock levando o throttling em conta."""
    if temp_c < 83:
        return clock_base
    # Perda de ~1.5% do clock por grau acima de 83 C
    perda = (temp_c - 83) * 0.015
    return int(clock_base * max(0.4, 1 - perda))

for t in [70, 80, 85, 90, 95]:
    c = clock_efetivo(t)
    flag = "  <-- throttling!" if c < 1410 else ""
    print(f"Temp {t}C -> clock {c} MHz{flag}")


### 4. Power Limit: o controle mais eficaz

Reduzir o **Power Limit** em 20–30% costuma custar só 5–10% de throughput. Usamos um **modelo
logístico** (curva realista): PL muito baixo sofre throttling e o ganho satura no topo — então
a **eficiência (imgs/J)** tem um ponto ótimo perto de **~60–75% do TDP**.


In [ ]:
# @title ⚙️ Benchmark energetico: throughput x eficiencia por Power Limit
# ============================================================================
# OBJETIVO: achar o PL de melhor eficiencia (imgs/J).
# Modelo logistico: throughput cresce e satura; consumo ~ 92% do limite.
# ============================================================================
import math

TDP = 250  # potencia maxima de referencia (W)

def throughput(pl, tdp=TDP):
    x = pl / tdp
    return 480 / (1 + math.exp(-6.0 * (x - 0.5)))   # curva logistica

resultados = []
print(f"{'PL (W)':>8} {'Imgs/s':>10} {'Pwr (W)':>10} {'Imgs/J':>10}")
print("-" * 42)
for pl in [100, 150, 200, 225, 250]:
    imgs = throughput(pl)
    pwr = pl * 0.92
    efic = imgs / pwr
    resultados.append((pl, imgs, pwr, efic))
    print(f"{pl:>8} {imgs:>10.1f} {pwr:>10.1f} {efic:>10.3f}")

melhor = max(resultados, key=lambda r: r[3])
print(f"\nPonto otimo: PL={melhor[0]}W (~{melhor[0]/TDP:.0%} do TDP), {melhor[3]:.3f} imgs/J")


### Prática 2 — visualizar o trade-off

O gráfico mostra **throughput** (linha da esquerda) e **eficiência** (linha da direita) em
função do Power Limit. O ponto ótimo de eficiência raramente é o de potência máxima.


In [ ]:
# @title 📊 Pratica 2: grafico throughput x eficiencia
# ============================================================================
# OBJETIVO: visualizar onde esta o melhor custo-beneficio energetico.
# ============================================================================
import matplotlib.pyplot as plt

pls = [r[0] for r in resultados]
imgs = [r[1] for r in resultados]
efics = [r[3] for r in resultados]

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(pls, imgs, "o-", color="#6366F1", label="Throughput (imgs/s)")
ax1.set_xlabel("Power Limit (W)")
ax1.set_ylabel("Throughput (imgs/s)", color="#6366F1")
ax1.tick_params(axis="y", labelcolor="#6366F1")

ax2 = ax1.twinx()
ax2.plot(pls, efics, "s--", color="#10B981", label="Eficiencia (imgs/J)")
ax2.set_ylabel("Eficiencia (imgs/J)", color="#10B981")
ax2.tick_params(axis="y", labelcolor="#10B981")

ax1.axvline(x=melhor[0], color="#F97316", linestyle=":", alpha=0.7)
ax1.set_title("Trade-off: Desempenho x Eficiencia Energetica")
plt.tight_layout()
plt.savefig("tradeoff_energia_dupla.png", dpi=120)
plt.show()
print(f"Melhor eficiencia em PL={melhor[0]}W.")


### 5. Controle programático com nvidia-ml-py (NVML)

O `nvidia-ml-py` lê e configura a GPU **sem parsing de texto**. Sem GPU, mostramos os mesmos
campos com valores de referência.


In [ ]:
# @title 🔌 Leitura via NVML (ou valores de referencia sem GPU)
# ============================================================================
# OBJETIVO: ler temperatura, potencia e faixa de Power Limit.
# ============================================================================
try:
    import pynvml
    pynvml.nvmlInit()
    h = pynvml.nvmlDeviceGetHandleByIndex(0)
    nome = pynvml.nvmlDeviceGetName(h)
    temp = pynvml.nvmlDeviceGetTemperature(h, pynvml.NVML_TEMPERATURE_GPU)
    pwr = pynvml.nvmlDeviceGetPowerUsage(h) / 1000
    pl = pynvml.nvmlDeviceGetPowerManagementLimit(h) / 1000
    pl_min, pl_max = pynvml.nvmlDeviceGetPowerManagementLimitConstraints(h)
    print(f"GPU: {nome} | {temp}C | {pwr:.1f}W / {pl:.1f}W")
    print(f"Power Limit suportado: {pl_min//1000}W - {pl_max//1000}W")
    pynvml.nvmlShutdown()
except Exception as e:
    print("[SIMULADO] NVML indisponivel:", e)
    print("GPU: Tesla T4 (referencia) | 67C | 68.5W / 70.0W")
    print("Power Limit suportado: 60W - 70W")


## Integração — juntando as duas partes

Só faz sentido medir energia **com a GPU exclusiva**: dois jobs juntos contaminam os dados.
Por isso o **lock vem primeiro**. Aqui, medimos a eficiência de um job que já detém o lock.


In [ ]:
# @title 🧩 Job energetico com lock (integra as duas partes)
# ============================================================================
# OBJETIVO: executar um 'treino' simulado DENTRO do lock e reportar a eficiencia.
# ============================================================================
if tentar_lock():
    print("Lock adquirido — medindo eficiencia com a GPU exclusiva...")
    time.sleep(0.5)
    pl_escolhido = melhor[0]
    imgs = throughput(pl_escolhido)
    efic = imgs / (pl_escolhido * 0.92)
    print(f"PL={pl_escolhido}W | {imgs:.0f} imgs/s | {efic:.3f} imgs/J")
    liberar_lock()
else:
    print("GPU ocupada — outro job esta medindo. Tente mais tarde.")


## 📝 Exercícios (5)

Resolva os 5 exercícios **neste notebook**. O valor está em **experimentar e explicar**.

---

**1) Sem lock.** Rode a célula da race condition com VRAM de 8 GB (em vez de 4). Quantos jobs
cabem? A OOM ainda acontece? O que isso ensina sobre planejar a VRAM?

**2) Lock não-bloqueante.** Modifique `tentar_lock()` para **desistir** após 3 tentativas e
imprimir "GPU ocupada, tentarei depois" — em vez de esperar para sempre.

**3) Fila com aging.** Um job de prioridade 3 pode nunca rodar (starvation). Descreva, em
pseudocódigo, como o **aging** (envelhecer o ticket) resolve isso.

**4) Power Limit.** No benchmark, qual PL deu a **melhor eficiência (imgs/J)**? O ganho de
throughput de subir de 200 W para 250 W compensa o aumento de consumo? Justifique.

**5) Integração.** Escreva um pequeno plano (5–8 linhas) para operar **1 GPU com 3 usuários**,
combinando o **lock/fila** e o **Power Limit** desta aula.

---

### Célula-esqueleto (complete para o exercício 2)

```python
# TODO: tentar_lock com no maximo 3 tentativas
# def tentar_lock_com_limite(max_tentativas=3):
#     ...
```
